# Welly AI RAG Notebook

Notebook นี้เป็นต้นแบบ RAG สำหรับโปรเจกต์ Welly AI  
โครงสร้างจะคล้ายไฟล์ตัวอย่างที่ให้มา แต่ปรับให้ใช้ knowledge และ outputs ของโปรเจกต์นี้โดยตรง

## แหล่งข้อมูลที่ใช้
- `../data/knowledge/standard_df.csv`
- `../data/knowledge/dga_standard_df.csv`
- `../data/knowledge/dga_rules_df.csv`
- `../data/knowledge/bmi_standard_df.csv`
- `../data/knowledge/bmi_rules_df.csv`
- `../data/knowledge/claim_rules_df.csv`
- `../data/knowledge/label_required_nutrients_df.csv`
- `../data/knowledge/serving_size_reference_df.csv`
- `../data/knowledge/user_health_knowledge.csv`
- `../outputs/food_dataset_with_risk.csv`
- `../outputs/full_knowledge_base.csv` (ถ้ามี)

## เป้าหมาย
- รวมข้อมูลเป็น documents
- สร้าง embeddings + FAISS vector store
- ทำ retrieval
- ใช้ LLM สรุปคำตอบจาก context ที่ค้นเจอ


In [4]:
# ติดตั้งแพ็กเกจที่จำเป็น (รันใน Colab หรือ notebook environment ที่ยังไม่มี library)
!pip -q install -U langchain langchain-community langchain-core langchain-text-splitters \
    langchain-huggingface langchain-groq faiss-cpu sentence-transformers rapidfuzz python-Levenshtein


In [5]:
%pip install -U langchain langchain-community langchain-core langchain-text-splitters langchain-huggingface langchain-groq faiss-cpu sentence-transformers rapidfuzz python-Levenshtein


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
import sys
print(sys.executable)

/usr/local/bin/python3


In [7]:
import rapidfuzz
import langchain_core
import langchain_huggingface
import langchain_community
import langchain_text_splitters
import langchain_groq

print("all imports ok")

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1
/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


all imports ok


## 1. Import Libraries

In [8]:
import os
import json
import re
from pathlib import Path

import pandas as pd

from rapidfuzz import process
from difflib import get_close_matches

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA


## 2. Load Secrets

ใช้ `LC_TOKEN` และ `GROQ_TOKEN` จาก Colab Secrets หรือ environment variables  
ถ้าไม่มี token จะยังสร้าง vector store และทดสอบ retrieval ได้ แต่จะยังใช้ LLM ตอบเต็มรูปไม่ได้


In [ ]:
LC_TOKEN = "LC_TOKEN"
GROQ_TOKEN = "GROQ_TOKEN"



# fallback ไปใช้ environment variable
LC_TOKEN = LC_TOKEN or os.environ.get("LC_TOKEN")
GROQ_TOKEN = GROQ_TOKEN or os.environ.get("GROQ_TOKEN")

if LC_TOKEN:
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_API_KEY"] = LC_TOKEN

if GROQ_TOKEN:
    os.environ["GROQ_API_KEY"] = GROQ_TOKEN

print("LC_TOKEN loaded:", LC_TOKEN is not None)
print("GROQ_TOKEN loaded:", GROQ_TOKEN is not None)


LC_TOKEN loaded: True
GROQ_TOKEN loaded: True


## 3. Locate Project Files

In [10]:
knowledge_dir = Path("../data/knowledge")
outputs_dir = Path("../outputs")

candidate_files = [
    knowledge_dir / "standard_df.csv",
    knowledge_dir / "dga_standard_df.csv",
    knowledge_dir / "dga_rules_df.csv",
    knowledge_dir / "bmi_standard_df.csv",
    knowledge_dir / "bmi_rules_df.csv",
    knowledge_dir / "claim_rules_df.csv",
    knowledge_dir / "label_required_nutrients_df.csv",
    knowledge_dir / "serving_size_reference_df.csv",
    knowledge_dir / "user_health_knowledge.csv",
    outputs_dir / "food_dataset_with_risk.csv",
    outputs_dir / "full_knowledge_base.csv",
]

file_status = pd.DataFrame({
    "file": [str(p) for p in candidate_files],
    "exists": [p.exists() for p in candidate_files]
})
file_status


,file,exists
0,../data/knowledge/standard_df.csv,True
1,../data/knowledge/dga_standard_df.csv,True
2,../data/knowledge/dga_rules_df.csv,True
3,../data/knowledge/bmi_standard_df.csv,True
4,../data/knowledge/bmi_rules_df.csv,True
5,../data/knowledge/claim_rules_df.csv,True
6,../data/knowledge/label_required_nutrients_df.csv,True
7,../data/knowledge/serving_size_reference_df.csv,True
8,../data/knowledge/user_health_knowledge.csv,True
9,../outputs/food_dataset_with_risk.csv,True


## 4. Load Tables

In [11]:
loaded_tables = {}

for path in candidate_files:
    if path.exists():
        try:
            loaded_tables[path.name] = pd.read_csv(path)
        except Exception as e:
            print(f"Failed to load {path.name}: {e}")

print("Loaded tables:", list(loaded_tables.keys()))
for name, df in loaded_tables.items():
    print(name, df.shape)


Loaded tables: ['standard_df.csv', 'dga_standard_df.csv', 'dga_rules_df.csv', 'bmi_standard_df.csv', 'bmi_rules_df.csv', 'claim_rules_df.csv', 'label_required_nutrients_df.csv', 'serving_size_reference_df.csv', 'user_health_knowledge.csv', 'food_dataset_with_risk.csv', 'full_knowledge_base.csv']
standard_df.csv (7, 6)
dga_standard_df.csv (8, 7)
dga_rules_df.csv (4, 4)
bmi_standard_df.csv (10, 9)
bmi_rules_df.csv (3, 5)
claim_rules_df.csv (3, 6)
label_required_nutrients_df.csv (9, 4)
serving_size_reference_df.csv (7, 5)
user_health_knowledge.csv (5000, 17)
food_dataset_with_risk.csv (2395, 20)
full_knowledge_base.csv (7423, 2)


## 5. Preview Tables

In [12]:
for name, df in loaded_tables.items():
    print("=" * 100)
    print(name)
    display(df.head(3))


standard_df.csv


,source_doc,category,metric,recommended_value,unit,note
0,media.pdf,daily_reference,Energy,2000,kcal/day,Thai RDI reference base
1,media.pdf,daily_reference,Total Fat,65,g/day,Thai RDI reference
2,media.pdf,daily_reference,Saturated Fat,20,g/day,Thai RDI reference


dga_standard_df.csv


,source_doc,category,metric,min_value,max_value,unit,target_group
0,DGA.pdf,Protein,protein_intake,1.2,1.6,g/kg/day,general
1,DGA.pdf,Dairy,dairy_servings,3.0,3.0,servings/day,2000_kcal_pattern
2,DGA.pdf,Vegetables,vegetable_servings,3.0,3.0,servings/day,2000_kcal_pattern


dga_rules_df.csv


...

,Patient_ID,Age,Gender,Height_cm,Weight_kg,BMI,BMI_Category,Blood_Pressure_Systolic,Blood_Pressure_Diastolic,Blood_Sugar_Level,Cholesterol_Level,Health_Profile_Summary,Recommended_Calories,Recommended_Protein,Recommended_Carbs,Recommended_Fats,Recommended_Meal_Plan
0,P00001,56,Other,163,66,24.84,Overweight,175,75,124,219,Overall moderate cardiometabolic risk. BMI: Ov...,2150,108,139,145,High-Protein Diet
1,P00002,69,Female,171,114,38.99,Obese Level 2,155,72,72,208,Overall moderate cardiometabolic risk. BMI: Ob...,1527,74,266,80,Balanced Diet
2,P00003,46,Female,172,119,40.22,Obese Level 2,137,101,145,171,Overall high cardiometabolic risk. BMI: Obese ...,2359,180,145,143,High-Protein Diet


food_dataset_with_risk.csv


,food_name,calories,fat,sat_fat,carbs,sugar,protein,fiber,cholesterol,sodium,sodium_pct_daily,cholesterol_pct_daily,sat_fat_pct_daily,fat_pct_daily,carbs_pct_daily,fiber_pct_daily,sugar_pct_meal_limit,risk_level,risk_label,chatbot_summary
0,cream cheese,51,5.0,2.9,0.8,0.5,0.9,0.0,14.6,0.016,0.0008,4.866667,14.5,7.692308,0.266667,0.0,5.0,Low,0,ยังไม่พบตัวชี้วัดที่เกินเกณฑ์เบื้องต้น
1,neufchatel cheese,215,19.4,10.9,3.1,2.7,7.8,0.0,62.9,0.300,0.0150,20.966667,54.5,29.846154,1.033333,0.0,27.0,Medium,1,โคเลสเตอรอลค่อนข้างสูง | ไขมันอิ่มตัวค่อนข้างสูง
2,requeijao cremoso light catupiry,49,3.6,2.3,0.9,3.4,0.8,0.1,0.0,0.000,0.0000,0.000000,11.5,5.538462,0.300000,0.4,34.0,Low,0,ยังไม่พบตัวชี้วัดที่เกินเกณฑ์เบื้องต้น


full_knowledge_base.csv


,source,text
0,standard_df,Nutrition standard metric Energy. Category dai...
1,standard_df,Nutrition standard metric Total Fat. Category ...
2,standard_df,Nutrition standard metric Saturated Fat. Categ...


## 6. Convert Tables to Documents

จะแปลงแต่ละ row ให้เป็นข้อความสั้น ๆ พร้อม metadata  
เพื่อให้ retrieval ดึงข้อมูลได้ตรงขึ้น


In [13]:
def row_to_text(table_name, row):
    table = table_name.lower()

    if table == "standard_df.csv":
        return (
            f"Nutrition standard. Source {row.get('source_doc', '')}. "
            f"Category {row.get('category', '')}. Metric {row.get('metric', '')}. "
            f"Recommended value {row.get('recommended_value', '')} {row.get('unit', '')}. "
            f"Target group {row.get('target_group', '')}. Note {row.get('note', '')}."
        )

    if table == "dga_standard_df.csv":
        return (
            f"DGA standard. Source {row.get('source_doc', '')}. Category {row.get('category', '')}. "
            f"Metric {row.get('metric', '')}. Min value {row.get('min_value', '')}. "
            f"Max value {row.get('max_value', '')} {row.get('unit', '')}. "
            f"Target group {row.get('target_group', '')}. Note {row.get('note', '')}."
        )

    if table == "dga_rules_df.csv":
        return (
            f"DGA rule. Source {row.get('source_doc', '')}. Rule name {row.get('rule_name', '')}. "
            f"Condition {row.get('condition', '')}. Unit {row.get('unit', '')}. "
            f"Note {row.get('note', '')}."
        )

    if table == "bmi_standard_df.csv":
        return (
            f"BMI standard. Source {row.get('source_doc', '')}. Category {row.get('category', '')}. "
            f"Metric {row.get('metric', '')}. Label {row.get('label', '')}. "
            f"Min value {row.get('min_value', '')}. Max value {row.get('max_value', '')}. "
            f"Unit {row.get('unit', '')}. Target group {row.get('target_group', '')}. "
            f"Note {row.get('note', '')}."
        )

    if table == "bmi_rules_df.csv":
        return (
            f"BMI rule. Source {row.get('source_doc', '')}. Rule name {row.get('rule_name', '')}. "
            f"Condition {row.get('condition', '')}. Unit {row.get('unit', '')}. "
            f"Note {row.get('note', '')}."
        )

    if table == "claim_rules_df.csv":
        return (
            f"Claim rule. Nutrient {row.get('nutrient', '')}. "
            f"Claim type {row.get('claim_type', '')}. Condition {row.get('condition', '')}. "
            f"Threshold value {row.get('threshold_value', '')} {row.get('unit', '')}. "
            f"Reference {row.get('reference', '')}."
        )

    if table == "label_required_nutrients_df.csv":
        return (
            f"Label required nutrient. Nutrient {row.get('nutrient', '')}. "
            f"Is required {row.get('is_required', '')}. Note {row.get('note', '')}."
        )

    if table == "serving_size_reference_df.csv":
        return (
            f"Serving size reference. Category {row.get('category', '')}. "
            f"Serving size {row.get('serving_size', '')} {row.get('unit', '')}. "
            f"Note {row.get('note', '')}."
        )

    if table == "user_health_knowledge.csv":
        return (
            f"User health knowledge. Patient ID {row.get('Patient_ID', row.get('patient_id', ''))}. "
            f"Age {row.get('Age', '')}. Gender {row.get('Gender', '')}. "
            f"Height {row.get('Height_cm', '')} cm. Weight {row.get('Weight_kg', '')} kg. "
            f"BMI {row.get('BMI', '')}. BMI category {row.get('BMI_Category', '')}. "
            f"Blood pressure systolic {row.get('Blood_Pressure_Systolic', '')}. "
            f"Blood pressure diastolic {row.get('Blood_Pressure_Diastolic', '')}. "
            f"Blood sugar {row.get('Blood_Sugar_Level', '')}. "
            f"Cholesterol {row.get('Cholesterol_Level', '')}. "
            f"Recommended calories {row.get('Recommended_Calories', '')}. "
            f"Recommended protein {row.get('Recommended_Protein', '')}. "
            f"Recommended carbs {row.get('Recommended_Carbs', '')}. "
            f"Recommended fats {row.get('Recommended_Fats', '')}. "
            f"Recommended meal plan {row.get('Recommended_Meal_Plan', '')}. "
            f"Health summary {row.get('Health_Profile_Summary', '')}."
        )

    if table == "food_dataset_with_risk.csv":
        return (
            f"Food risk record. Food name {row.get('food_name', '')}. "
            f"Calories {row.get('calories', '')}. Fat {row.get('fat', '')}. "
            f"Saturated fat {row.get('sat_fat', '')}. Carbs {row.get('carbs', '')}. "
            f"Sugar {row.get('sugar', '')}. Protein {row.get('protein', '')}. "
            f"Fiber {row.get('fiber', '')}. Cholesterol {row.get('cholesterol', '')}. "
            f"Sodium {row.get('sodium', '')}. Risk level {row.get('risk_level', '')}. "
            f"Chatbot summary {row.get('chatbot_summary', '')}."
        )

    if table == "full_knowledge_base.csv":
        return (
            f"Full knowledge base record. Source {row.get('source', '')}. "
            f"Text {row.get('text', '')}."
        )

    # fallback
    joined = ". ".join([f"{col}: {row.get(col, '')}" for col in row.index])
    return f"{table_name} record. {joined}"


In [14]:
documents = []

for table_name, df in loaded_tables.items():
    for idx, row in df.iterrows():
        text = row_to_text(table_name, row)
        doc = Document(
            page_content=text,
            metadata={
                "table": table_name,
                "row_index": int(idx)
            }
        )
        documents.append(doc)

print("Total documents:", len(documents))
documents[:3]


Total documents: 14869


[Document(metadata={'table': 'standard_df.csv', 'row_index': 0}, page_content='Nutrition standard. Source media.pdf. Category daily_reference. Metric Energy. Recommended value 2000 kcal/day. Target group . Note Thai RDI reference base.'),
 Document(metadata={'table': 'standard_df.csv', 'row_index': 1}, page_content='Nutrition standard. Source media.pdf. Category daily_reference. Metric Total Fat. Recommended value 65 g/day. Target group . Note Thai RDI reference.'),
 Document(metadata={'table': 'standard_df.csv', 'row_index': 2}, page_content='Nutrition standard. Source media.pdf. Category daily_reference. Metric Saturated Fat. Recommended value 20 g/day. Target group . Note Thai RDI reference.')]

## 7. Optional Text Splitting

ถ้าเอกสารยาวมากสามารถ split ได้  
แต่กรณีนี้แต่ละ row สั้นอยู่แล้ว จึง split แบบเบา ๆ หรือข้ามได้


In [15]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

split_docs = splitter.split_documents(documents)
print("Split documents:", len(split_docs))
split_docs[:2]


Split documents: 24869


[Document(metadata={'table': 'standard_df.csv', 'row_index': 0}, page_content='Nutrition standard. Source media.pdf. Category daily_reference. Metric Energy. Recommended value 2000 kcal/day. Target group . Note Thai RDI reference base.'),
 Document(metadata={'table': 'standard_df.csv', 'row_index': 1}, page_content='Nutrition standard. Source media.pdf. Category daily_reference. Metric Total Fat. Recommended value 65 g/day. Target group . Note Thai RDI reference.')]

## 8. Build Embeddings + Vector Store

In [16]:
# ใช้ multilingual model เพื่อรองรับคำถามภาษาไทยและอังกฤษ
model_name = "intfloat/multilingual-e5-base"

try:
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
except Exception:
    device = "cpu"

print("Embedding device:", device)

embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True, "batch_size": 32}
)

index_dir = "faiss_welly_index"

if os.path.exists(index_dir):
    vectorstore = FAISS.load_local(index_dir, embeddings, allow_dangerous_deserialization=True)
    print("Loaded existing FAISS index")
else:
    vectorstore = FAISS.from_documents(split_docs, embeddings)
    vectorstore.save_local(index_dir)
    print("Built and saved new FAISS index")


Embedding device: cpu


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 10285.98it/s]
XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded existing FAISS index


## 9. Quick Retrieval Test

In [17]:
test_queries = [
    "BMI ปกติคือเท่าไร",
    "โซเดียมต่อวันควรไม่เกินเท่าไร",
    "น้ำตาลต่อมื้อควรไม่เกินเท่าไร",
    "cream cheese เสี่ยงไหม",
    "อาหารที่มีคอเลสเตอรอลสูง"
]

for q in test_queries:
    print("=" * 100)
    print("Query:", q)
    results = vectorstore.similarity_search(q, k=3)
    for i, doc in enumerate(results, start=1):
        print(f"{i}. [{doc.metadata.get('table')}] {doc.page_content[:300]}")
    print()


Query: BMI ปกติคือเท่าไร
1. [bmi_standard_df.csv] BMI standard. Source document-20210831192536.pdf. Category BMI. Metric bmi_category. Label Normal. Min value 18.5. Max value 22.99. Unit kg/m^2. Target group asian_adults. Note น้ำหนักปกติ.
2. [bmi_standard_df.csv] BMI standard. Source document-20210831192536.pdf. Category BMI. Metric bmi_category. Label Underweight. Min value -inf. Max value 18.49. Unit kg/m^2. Target group asian_adults. Note น้ำหนักต่ำกว่าเกณฑ์.
3. [bmi_standard_df.csv] BMI standard. Source document-20210831192536.pdf. Category Waist Circumference. Metric waist_risk_threshold. Label Normal. Min value nan. Max value 90.0. Unit cm. Target group male. Note ผู้ชายควรมีเส้นรอบเอวไม่เกิน 90 ซม..

Query: โซเดียมต่อวันควรไม่เกินเท่าไร
1. [dga_standard_df.csv] DGA standard. Source DGA.pdf. Category Sodium. Metric sodium_limit. Min value nan. Max value 2300.0 mg/day. Target group age_14_plus. Note .
2. [full_knowledge_base.csv] Full knowledge base record. Source dga_standard_df

## 10. Setup Retriever

In [18]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})


## 11. Setup LLM and Prompt

In [19]:
prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template="""คุณคือผู้ช่วยด้านสุขภาพและโภชนาการของโปรเจกต์ Welly AI

ตอบคำถามโดยอ้างอิงจากข้อมูลใน context เท่านั้น
ถ้า context ไม่พอ ให้ตอบตามตรงว่าไม่พบข้อมูลเพียงพอ
พยายามตอบเป็นภาษาไทยที่เข้าใจง่ายและกระชับ

Context:
{context}

Question:
{question}

Answer:
"""
)

llm = None
qa_chain = None

if GROQ_TOKEN:
    llm = ChatGroq(
        model="llama-3.1-8b-instant",
        max_tokens=500,
        temperature=0.2
    )

    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=retriever,
        chain_type="stuff",
        chain_type_kwargs={"prompt": prompt_template},
        return_source_documents=True
    )
    print("LLM + RetrievalQA ready")
else:
    print("No GROQ_TOKEN found: retrieval-only mode")


LLM + RetrievalQA ready


## 12. Ask Questions

In [20]:
def ask_welly_rag(question: str):
    if qa_chain is not None:
        result = qa_chain.invoke({"query": question})
        answer = result["result"]
        sources = result.get("source_documents", [])
        return answer, sources

    # fallback retrieval-only
    docs = retriever.invoke(question)
    answer = "ยังไม่มี LLM token ระบบจะแสดงเฉพาะ context ที่ค้นเจอ"
    return answer, docs


In [26]:
demo_questions = [
    # "BMI ปกติคือเท่าไร",
    # "โซเดียมต่อวันควรไม่เกินเท่าไร",
    # "น้ำตาลต่อมื้อควรไม่เกินเท่าไร",
    # "cream เสี่ยงไหม",
    # "อาหารที่มีคอเลสเตอรอลสูงมีตัวอย่างอะไรบ้าง",
    # "อยากกินครีมซีสอะดีไหม แล้วมีค่าอะไรเท่าบ้าง",
    "หมาอร่อยไหม",
    "อาหารหมามีประโยชน์ไหม"
]

for q in demo_questions:
    print("=" * 100)
    print("Question:", q)
    answer, sources = ask_welly_rag(q)
    print("Answer:", answer)
    # print("Sources:")
    # for i, doc in enumerate(sources[:3], start=1):
    #     print(f"{i}. [{doc.metadata.get('table')}] {doc.page_content[:250]}")
    # print()


Question: หมาอร่อยไหม
Answer: ไม่พบข้อมูลเพียงพอเกี่ยวกับ "หมาอร่อยไหม" ในฐานข้อมูลของเรา
Question: อาหารหมามีประโยชน์ไหม
Answer: อาหารหมามีประโยชน์ แต่ควรควบคุมปริมาณการบริโภค เนื่องจากมีไขมันอิ่มตัวและโคเลสเตอรอลสูง ซึ่งอาจส่งผลเสียต่อสุขภาพ หากบริโภคในปริมาณมากเกินไป


## 13. Optional Utility: Food Name Search

ช่วยหาชื่ออาหารที่ใกล้เคียงกรณีพิมพ์ไม่ตรงเป๊ะ


In [22]:
def suggest_food_names(user_text, food_df):
    if "food_name" not in food_df.columns:
        return []

    food_names = food_df["food_name"].dropna().astype(str).unique().tolist()
    q = user_text.strip()

    fuzzy = process.extract(q, food_names, limit=5)
    close = get_close_matches(q, food_names, n=5, cutoff=0.4)

    result = []
    for item in fuzzy:
        result.append(item[0])
    for item in close:
        if item not in result:
            result.append(item)

    return result[:5]

if "food_dataset_with_risk.csv" in loaded_tables:
    food_df = loaded_tables["food_dataset_with_risk.csv"]
    print(suggest_food_names("cream chese", food_df))


['cream cheese', 'cream cheese low fat', 'cream cheese fat free', 'baked potato with sour cream', 'cream of chicken soup']


## 14. Summary

Notebook นี้ทำหน้าที่เป็น RAG prototype สำหรับ Welly AI โดย:
- รวม knowledge และ outputs ของโปรเจกต์เป็น documents
- สร้าง embeddings และ FAISS vector store
- ทำ retrieval จากคำถามของผู้ใช้
- ใช้ LLM (ถ้ามี token) เพื่อสรุปคำตอบจาก context ที่ค้นเจอ

ถ้ายังไม่มี token ก็ยังใช้ในโหมด retrieval-only เพื่อทดสอบการค้นข้อมูลได้
